# SGDense-3DGS — segmentation-rendering performance branch

Tests the `perf/seg-render` branch:

1. **Correctness** — the `dL_dalpha` seg-folding guard (`tests/test_seg_backward.py`).
2. **Perf on the current code** — a short SGDense run with the phase profiler, so you can
   read `render` / `backward` / `optimizer` costs directly.
3. **The `--seg_render_interval K` lever** — run again with `K=8` and compare.

Changes under test: the second backward pass for the densification trigger is gone
(item 1, biggest), the backward seg `dL_dalpha` path uses two scalars instead of two
32-float register arrays (item 2), and seg rendering can be decimated (item 3, opt-in).
See `docs/09-iteration-performance.md`.

> Runtime: **GPU** (same known-good runtime as your other notebooks).
> Expects the Replica `office_0/Sequence_1` dataset (with `semantic_class/`) in the same
> Drive location as `Run_Experiments.ipynb`.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## 2. Get the `perf/seg-render` branch

Parent repo **and** the seg-aware rasterizer submodule both need the branch. The
submodule is a separate repo, so we check it out explicitly and confirm the commit.

In [ ]:
import os

REPO_URL = "https://github.com/satyaba/gaussian-splatting.git"
BRANCH   = "perf/seg-render"
REPO_DIR = "/content/gaussian-splatting"
RAST_DIR = os.path.join(REPO_DIR, "submodules", "diff-gaussian-rasterization")

if not os.path.isdir(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch --all -q
    !git -C {REPO_DIR} checkout -q {BRANCH}
    !git -C {REPO_DIR} pull --ff-only -q

os.chdir(REPO_DIR)

# Init submodules (rasterizer + simple-knn), then make sure the rasterizer is on
# the perf branch commit (the recorded SHA), not the old seg-encoding tip.
!git submodule update --init --recursive submodules/diff-gaussian-rasterization submodules/simple-knn
!git -C {RAST_DIR} fetch -q origin {BRANCH}
!git -C {RAST_DIR} checkout -q origin/{BRANCH}

# MUST print a commit whose subject mentions "photo-only 2D-mean grad" (parent 31a262a).
print("parent    :", end=" "); !git -C {REPO_DIR} log --oneline -1
print("rasterizer:", end=" "); !git -C {RAST_DIR} log --oneline -1
print("cwd:", os.getcwd())

## 3. GPU / CUDA check

In [ ]:
!nvidia-smi -L

import re, shutil, subprocess, warnings
import torch

print(f"PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | "
          f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

def _nvcc_cuda():
    exe = shutil.which("nvcc")
    if exe is None:
        return None
    out = subprocess.run([exe, "--version"], capture_output=True, text=True).stdout
    m = re.search(r"release (\d+)\.(\d+)", out)
    return f"{m.group(1)}.{m.group(2)}" if m else None

torch_cuda, nvcc_cuda = torch.version.cuda, _nvcc_cuda()
print(f"  torch built for CUDA {torch_cuda} | nvcc {nvcc_cuda or 'NOT FOUND'}")
if nvcc_cuda is None:
    warnings.warn("nvcc not found — CUDA extensions cannot be built.")
elif torch_cuda and torch_cuda.split(".")[0] != nvcc_cuda.split(".")[0]:
    raise RuntimeError(f"CUDA major mismatch: torch {torch_cuda} vs nvcc {nvcc_cuda}.")

In [ ]:
!pip install -q plyfile opencv-python tqdm joblib lpips torchmetrics tensorboard

## 4. Configure paths

Same layout as `experiments/runs.py`: `<HOME_DIR>/Dataset/<dataset>` with
`images/`, `sparse/0/` and `semantic_class/`.

In [ ]:
import os
from pathlib import Path

HOME_DIR = "/content/drive/MyDrive/Magister/Thesis/Script"   # <-- adjust if different
DATASET_NAME = "office_0/Sequence_1"

DATASET_DIR = Path(HOME_DIR) / "Dataset" / DATASET_NAME
SEG_DIR = DATASET_DIR / "semantic_class"
OUT_ROOT = Path(HOME_DIR) / "output"

for p in (DATASET_DIR / "images", DATASET_DIR / "sparse" / "0", SEG_DIR):
    assert p.exists(), f"missing: {p}"
print("dataset:", DATASET_DIR)
print("seg    :", SEG_DIR, f"({len(list(SEG_DIR.glob('*.png')))} PNGs)")

## 5. Build the seg-aware rasterizer (required — CUDA changed)

In [ ]:
# <cstdint> guard for newer CUDA/libstdc++ (idempotent; same patch the harness applies).
for f in ["submodules/diff-gaussian-rasterization/cuda_rasterizer/rasterizer_impl.h",
          "submodules/simple-knn/cuda_utils.h",
          "submodules/simple-knn/simple_knn.cu"]:
    if os.path.isfile(f):
        txt = open(f).read()
        if "include <cstdint>" not in txt:
            open(f, "w").write("#include <cstdint>\n" + txt)
            print("patched", f)

!pip install ./submodules/diff-gaussian-rasterization --no-build-isolation 2>&1 | tail -5
!pip install ./submodules/simple-knn --no-build-isolation 2>&1 | tail -3

import glob
sos = glob.glob("submodules/diff-gaussian-rasterization/**/*.so", recursive=True)
print("\nbuilt:", sos)
assert sos, "rasterizer .so not found — build failed"

import diff_gaussian_rasterization as dgr
print("import OK from:", dgr.__file__)

## 6. Correctness sanity — seg still drives geometry

`tests/test_seg_backward.py` is the doc-01 §2 guard: a seg-only loss must produce
nonzero `xyz`/`opacity`/`scaling` gradients (via the shared `dL_dalpha`) and exactly
zero color gradients. If the scalar-`dL_dalpha` refactor broke the folding, this fails.

In [ ]:
!python tests/test_seg_backward.py

## 7. Phase profile — default (every iteration, `K=1`)

A short run with the profiler enabled. We profile a window just past the seg warmup
(`seg_warmup_iters=2000`), where the old second backward used to run.

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

ITERATIONS   = 2600
PROFILE_WINDOW = "2400:2599"

def run_sgdense(tag, extra=()):
    out_dir = OUT_ROOT / f"perf_{tag}"
    out_dir.mkdir(parents=True, exist_ok=True)
    profile_out = str(out_dir / "iter_profile.json")
    cmd = [sys.executable, "train.py",
           "-s", str(DATASET_DIR), "-m", str(out_dir),
           "--eval", "--disable_viewer", "--quiet",
           "--iterations", str(ITERATIONS),
           "--segmentation_path", str(SEG_DIR),
           "--num_segmentation_classes", "99",
           *extra]
    env = dict(os.environ,
               SGDENSE_PROFILE_ITERS=PROFILE_WINDOW,
               SGDENSE_PROFILE_OUT=profile_out)
    print(f"\n=== run {tag}: {' '.join(extra) or '(default)'} ===")
    t0 = time.time()
    subprocess.run(cmd, cwd=REPO_DIR, env=env, check=True)
    dt = time.time() - t0
    print(f"wall: {dt:.1f}s ({dt / ITERATIONS * 1000:.1f} ms/iter avg)")
    return profile_out, dt

def show_profile(path, title):
    g = json.load(open(path))["summary"]["groups"].get("all", {})
    print(f"\n{title}")
    print(f"  {'phase':<10} {'mean_ms':>9} {'p90_ms':>9}")
    for ph in ["render", "loss", "backward", "optimizer", "densify", "misc", "total"]:
        m = g.get(ph, {})
        if "mean_ms" in m:
            print(f"  {ph:<10} {m['mean_ms']:>9.2f} {m.get('p90_ms', float('nan')):>9.2f}")

json_k1, wall_k1 = run_sgdense("K1")
show_profile(json_k1, "K=1 (every iteration)")

## 8. Phase profile — seg-render interval `K=8`

Off-iterations skip the 32-channel seg compositing and the decoder, so they cost
≈ vanilla. **This changes training dynamics** (fewer decoder updates / seg→geometry
gradients) — treat it as an ablation, not a free win.

In [ ]:
json_k8, wall_k8 = run_sgdense("K8", extra=("--seg_render_interval", "8"))
show_profile(json_k8, "K=8 (seg every 8th iteration)")

print(f"\nWall-clock: K=1 {wall_k1:.1f}s  |  K=8 {wall_k8:.1f}s  "
      f"({wall_k1 / wall_k8:.2f}x)")

### Reading the numbers

- **`backward`** should now be the single backward only (the old `autograd.grad` pass
  is gone). Compare against any profile you saved before this branch.
- **`render`** drops, on average, by roughly `seg_extra * (K-1)/K` at `K=8`.
- **`optimizer` / `misc`** are unchanged.
- Sanity: PSNR shouldn't collapse at `K=1`; `K=8` is the ablation to watch.

## 9. (Optional) Full A/B against vanilla 3DGS

The `experiments.profile` harness clones upstream 3DGS, rebuilds the stock rasterizer,
runs both methods, and prints the per-phase SGDense-vs-vanilla table. It **swaps the
installed rasterizer**, so run it *after* the cells above (it leaves the stock
rasterizer installed; re-run cell 5 to switch back to the seg-aware one).

In [ ]:
# Dry run first to see the two commands it would launch:
!python -m experiments.profile --source "{DATASET_DIR}" --home-dir "{HOME_DIR}" \
    --iterations 2600 --profile-iters 2400:2599 --seg-num-classes 99 --dry-run

# Then (uncomment) the real thing — expect ~2x the wall time of the cells above:
# !python -m experiments.profile --source "{DATASET_DIR}" --home-dir "{HOME_DIR}" \
#     --iterations 2600 --profile-iters 2400:2599 --seg-num-classes 99